In [5]:
# === TODO-EN-UNO: genera faces_detections.csv si falta + Talking-Face SRT/CSV ===
# Config de entradas/salidas
from pathlib import Path
VIDEO = "../data/video.mp4"         # <-- tu vídeo (unificado)
OUT_DIR = Path("diarization_elements")
OUT_DIR.mkdir(exist_ok=True)

DETECTIONS_CSV = OUT_DIR / "faces_detections.csv"   # detecciones InsightFace
CSV_TALKING   = OUT_DIR / "talking_faces.csv"       # intervalos talking-face
SRT_OUT       = OUT_DIR / "talking_faces.srt"       # salida SRT

# --- Dependencias mínimas (auto-install si faltan) ---
import sys, subprocess, importlib, os
from importlib import import_module

def need(m):
    try:
        importlib.import_module(m); return False
    except Exception:
        return True

def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", *pkgs], check=True)

# Núcleo visión/datos
if need("numpy"):  pip_install("numpy<2")
if need("opencv"): pip_install("opencv-python")
if need("pandas"): pip_install("pandas")
if need("tqdm"):   pip_install("tqdm")

# Para detecciones y clustering (si hay que crear el CSV)
if not DETECTIONS_CSV.exists():
    if need("insightface"):  pip_install("insightface==0.7.3")
    if need("onnxruntime"):  pip_install("onnxruntime==1.17.3")
    if need("sklearn"):      pip_install("scikit-learn")

# Para audio-gate y FaceMesh
if need("mediapipe"): pip_install("mediapipe")
if need("moviepy"):   pip_install("moviepy==1.0.3")
if need("librosa"):   pip_install("librosa", "soundfile")

# --- Imports una vez instalados ---
import cv2, numpy as np, pandas as pd
from tqdm import tqdm

# Utilidades comunes
def fmt_ts(seconds: float) -> str:
    ms_total = int(round(seconds * 1000))
    hh, rem = divmod(ms_total, 3600_000)
    mm, rem = divmod(rem, 60_000)
    ss, ms = divmod(rem, 1000)
    return f"{hh:02d}:{mm:02d}:{ss:02d},{ms:03d}"

PAD2 = True
def face_label(fid: int) -> str:
    return f"FACE_{fid:02d}" if PAD2 else f"FACE_{fid}"

# ========== (1) CREAR faces_detections.csv SI NO EXISTE ==========
if not DETECTIONS_CSV.exists():
    print("No existe faces_detections.csv -> generándolo con InsightFace + DBSCAN...")

    FaceAnalysis = import_module("insightface.app.face_analysis").FaceAnalysis
    from sklearn.cluster import DBSCAN

    SAMPLE_FPS_DET = 5
    DBSCAN_EPS = 0.45
    MIN_SAMPLES = 3

    cap = cv2.VideoCapture(str(VIDEO))
    if not cap.isOpened():
        raise SystemExit(f"No se pudo abrir el vídeo: {VIDEO}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    step = max(1, int(round(fps / SAMPLE_FPS_DET)))

    try:
        app = FaceAnalysis(name="buffalo_l")
    except Exception:
        FaceAnalysis = import_module("insightface.app.face_analysis").FaceAnalysis
        app = FaceAnalysis(name="buffalo_l")

    try:
        app.prepare(ctx_id=0, det_size=(640,640))   # GPU si disponible
    except Exception:
        app.prepare(ctx_id=-1, det_size=(640,640))  # CPU si no hay GPU

    embeds, times, bboxes = [], [], []
    f = 0
    ok, frame = cap.read()
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    with tqdm(total=total, desc="Detecciones+Embeddings") as pbar:
        while ok:
            if f % step == 0:
                t = f / fps
                faces = app.get(frame) or []
                for fa in faces:
                    emb = getattr(fa, "normed_embedding", None)
                    bbox = getattr(fa, "bbox", None)
                    if emb is None or bbox is None:
                        continue
                    embeds.append(emb)
                    times.append(t)
                    bboxes.append(np.asarray(bbox, dtype=int).tolist())
            ok, frame = cap.read(); f += 1; pbar.update(1)
    cap.release()

    if not embeds:
        raise SystemExit("No se detectaron caras para generar detecciones. Revisa el vídeo o baja det_size.")

    X = np.vstack(embeds)
    cl = DBSCAN(eps=DBSCAN_EPS, min_samples=MIN_SAMPLES, metric="cosine").fit(X)
    labels = cl.labels_

    det_df = pd.DataFrame({
        "time_s": times,
        "face_id": labels,
        "x1": [bb[0] for bb in bboxes],
        "y1": [bb[1] for bb in bboxes],
        "x2": [bb[2] for bb in bboxes],
        "y2": [bb[3] for bb in bboxes],
    })
    det_df.to_csv(DETECTIONS_CSV, index=False)
    print("Detecciones guardadas en:", DETECTIONS_CSV.resolve())

# ========== (2) TALKING-FACE por labios + (opcional) audio gate ==========
# Parámetros talking-face (tomados de tu script “original”)
SAMPLE_FPS_MESH = 10      # debe ir en la misma escala que las detecciones
GAP_MERGE_S = 0.5         # fusiona huecos <= 0.5 s
USE_AUDIO_GATE = True
AUDIO_GATE_PCTL = 60
MOUTH_THRESH_DELTA = 0.02
# Más estricto: sube MOUTH_THRESH_DELTA (+0.02) y/o AUDIO_GATE_PCTL=70–75
# Más permisivo: baja MOUTH_THRESH_DELTA o USE_AUDIO_GATE=False
# Menos cortes: sube GAP_MERGE_S a 0.8–1.0

# Landmarks labios (FaceMesh)
import mediapipe as mp
UP_C, LO_C, LEFT, RIGHT = 13, 14, 61, 291

# Cargar detecciones
df = pd.read_csv(DETECTIONS_CSV)
print("CARGANDO DETECCIONES DE CARAS")

# Normaliza nombres si vienen distintos
if "face_id" not in df.columns:
    if "face_id_or_-1" in df.columns:
        df = df.rename(columns={"face_id_or_-1":"face_id"})

if "time_s" not in df.columns:
    # permitir frame_idx
    if "frame_idx" in df.columns:
        cap = cv2.VideoCapture(str(VIDEO))
        fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
        cap.release()
        df["time_s"] = df["frame_idx"].astype(int) / float(fps)
    else:
        raise SystemExit("El CSV de detecciones debe tener 'time_s' o 'frame_idx'.")

# Filtrar caras válidas y comprobar bbox
df = df[(df["face_id"] >= 0)].copy()
for c in ["x1","y1","x2","y2"]:
    if c not in df.columns:
        raise SystemExit("Faltan columnas de bbox x1,y1,x2,y2 en detecciones.")
df["time_s"] = df["time_s"].astype(float)
df["face_id"] = df["face_id"].astype(int)

# Preparar video random-access y agrupación por frame
cap = cv2.VideoCapture(str(VIDEO))
if not cap.isOpened():
    raise SystemExit(f"No se pudo abrir el vídeo: {VIDEO}")
fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
df["frame_idx"] = (df["time_s"] * fps).round().astype(int)
groups = df.groupby("frame_idx")

# MediaPipe FaceMesh
mp_face = mp.solutions.face_mesh
mesh = mp_face.FaceMesh(static_image_mode=True, max_num_faces=1, refine_landmarks=True)

def mouth_open_ratio(img_bgr, box):
    h, w = img_bgr.shape[:2]
    x1,y1,x2,y2 = [int(np.clip(v, 0, max(1, lim-1))) for v,lim in zip(box, [w,h,w,h])]
    if x2<=x1 or y2<=y1:
        return None
    roi = img_bgr[y1:y2, x1:x2]
    if roi.size == 0:
        return None
    rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)
    res = mesh.process(rgb)
    if not res.multi_face_landmarks:
        return None
    lm = res.multi_face_landmarks[0].landmark
    def pt(i): return np.array([lm[i].x*(x2-x1), lm[i].y*(y2-y1)], dtype=np.float32)
    up, lo, L, R = pt(UP_C), pt(LO_C), pt(LEFT), pt(RIGHT)
    mouth_h = np.linalg.norm(up - lo)
    mouth_w = np.linalg.norm(L - R) + 1e-6
    return float(mouth_h / mouth_w)

# Calcular mouth_ratio por detección
rat_s = pd.Series(index=df.index, dtype="float32")
hit_s = pd.Series(False, index=df.index)

for frame_idx, g in tqdm(groups, desc="FaceMesh labios"):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
    ok, frame = cap.read()
    if not ok:
        continue
    for ridx in g.index:
        r = df.loc[ridx]
        rati = mouth_open_ratio(frame, (r.x1, r.y1, r.x2, r.y2))
        if rati is not None:
            rat_s.at[ridx] = rati
            hit_s.at[ridx] = True

df["mouth_ratio"] = rat_s.values
df = df[hit_s.values].copy()

# Umbral adaptativo por cara
thr_by_face = (df.groupby("face_id")["mouth_ratio"].median()).to_dict()
df["thr_face"] = df["face_id"].map(thr_by_face)
df["is_talking_mouth"] = df["mouth_ratio"] > (df["thr_face"] + MOUTH_THRESH_DELTA)

# --- Gate de audio (librosa) ---
import librosa

def audio_energy_envelope(video_path, sr=16000, hop_s=0.05):
    y, sr = librosa.load(video_path, sr=sr, mono=True)
    hop = max(1, int(sr * hop_s))
    frame_rms = librosa.feature.rms(y=y, frame_length=2*hop, hop_length=hop, center=True).flatten()
    t = np.arange(len(frame_rms)) * (hop / sr)
    return t, frame_rms

talk_gate = None
if USE_AUDIO_GATE:
    try:
        t_env, env = audio_energy_envelope(str(VIDEO), sr=16000, hop_s=0.05)
        if t_env is not None:
            thr = np.percentile(env, AUDIO_GATE_PCTL)
            talk_gate = (t_env, env, float(thr))
    except Exception as e:
        print("Aviso: no se pudo calcular energía de audio, continuo sin gate.", e)

def is_voiced(t):
    if talk_gate is None:
        return True
    t_env, env, thr = talk_gate
    if len(t_env) < 2:
        return True
    dt = (t_env[1] - t_env[0]) if (t_env[1] - t_env[0]) > 0 else 1e-6
    idx = int(np.clip(round(t / dt), 0, len(env)-1))
    return env[idx] >= thr

df["is_talking"] = df.apply(lambda r: bool(r.is_talking_mouth and is_voiced(r.time_s)), axis=1)

# Construir segmentos por FACE (fusionando huecos)
from collections import defaultdict
def build_segments(times_bool, gap=0.6):
    if not times_bool:
        return []
    times_bool.sort()
    active = [t for t, v in times_bool if v]
    if not active:
        return []
    segs, s, prev = [], active[0], active[0]
    for t in active[1:]:
        if t - prev > gap:
            segs.append((s, prev)); s = t
        prev = t
    segs.append((s, prev))
    return segs

segments = []
for fid, g in df.groupby("face_id"):
    tb = list(zip(g["time_s"].tolist(), g["is_talking"].tolist()))
    for s,e in build_segments(tb, gap=GAP_MERGE_S):
        if e <= s:
            e = s + 0.2
        segments.append((int(fid), float(s), float(e)))

# Guardar resultados
segments.sort(key=lambda x: (x[1], x[2], x[0]))

# CSV con intervalos talking-face
seg_df = pd.DataFrame(segments, columns=["face_id","start_s","end_s"])
seg_df.to_csv(CSV_TALKING, index=False)
print(f"CSV 'talking faces' escrito: {CSV_TALKING.resolve()} | Filas: {len(seg_df)}")

# SRT con intervalos talking-face
with open(SRT_OUT, "w", encoding="utf-8") as f:
    for i, (fid, s, e) in enumerate(segments, 1):
        f.write(f"{i}\n")
        f.write(f"{fmt_ts(s)} --> {fmt_ts(e)}\n")
        f.write(f"{face_label(fid)}\n\n")

print(f"SRT 'talking faces' escrito: {SRT_OUT.resolve()} | Cues: {len(segments)}")
# ============================================================================


No existe faces_detections.csv -> generándolo con InsightFace + DBSCAN...


c:\Users\carlos.basallote\.conda\envs\torch-cu124\Lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:121: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\carlos.basallote/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\carlos.basallote/.insightface\models\buffalo_l\2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\carlos.basallote/.insightface\models\buffalo_l\det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\carlos.basallote/.insightface\models\buffalo_l\genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\carlos.basallote/.insightface\models\buffalo_

Detecciones+Embeddings: 100%|██████████| 24192/24192 [1:19:34<00:00,  5.07it/s]


Detecciones guardadas en: C:\Users\carlos.basallote\Desktop\TFM\TFM\code\diarization_elements\faces_detections.csv
CARGANDO DETECCIONES DE CARAS


FaceMesh labios: 100%|██████████| 4130/4130 [02:15<00:00, 30.39it/s]
C:\Users\carlos.basallote\AppData\Local\Temp\ipykernel_28880\2686122541.py:225: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(video_path, sr=sr, mono=True)
C:\Users\carlos.basallote\AppData\Roaming\Python\Python311\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


CSV 'talking faces' escrito: C:\Users\carlos.basallote\Desktop\TFM\TFM\code\diarization_elements\talking_faces.csv | Filas: 428
SRT 'talking faces' escrito: C:\Users\carlos.basallote\Desktop\TFM\TFM\code\diarization_elements\talking_faces.srt | Cues: 428
